# The post-mortem agent — cases

For every watchlist name and every tail surprise (actual landed at
`actual_quantile` ≤ 0.10 or ≥ 0.90), the post-mortem classifies why
(`src/postmortem.py`):

| label | when |
|---|---|
| `within-expected-dispersion` | close landed mid-distribution — the flag was noise |
| `catalyst:earnings` | the name reported within ±1 day (curated calendar) |
| `catalyst:unlisted` | LLM identified a real company event the calendar missed |
| `regime-move` | the correlated cluster moved together |
| `data-issue` | a split / stale-price / thin-volume artefact |
| `unexplained` | tail move, idiosyncratic, no scheduled catalyst |

Deterministic guardrails decide the clear cases; the LLM only sees the
genuinely ambiguous idiosyncratic tails.


In [1]:
import os
if "src" not in os.listdir("."):
    os.chdir("..")

In [2]:
import json, pandas as pd
from pathlib import Path
from src import config
eps = pd.DataFrame(json.loads(l) for l in
                   (config.MEMORY_DIR/"episodes.jsonl").read_text().splitlines() if l.strip())
print(len(eps), "episodes across", eps["asof"].nunique(), "sessions")
eps["classification"].value_counts()

48 episodes across 3 sessions


classification
regime-move                   22
unexplained                    9
within-expected-dispersion     9
catalyst:earnings              6
catalyst:unlisted              2
Name: count, dtype: int64

## 1 · One worked case per label

In [3]:
from src.postmortem import sector_move, known_catalyst, data_sanity
def show(ep):
    t, a = ep["ticker"], ep["asof"]
    print(f"=== {t}  {a}  ({ep['setup']}) ===")
    print(f"forecast median {ep['forecast_median']:+.2%} | actual {ep['actual']:+.2%} "
          f"| landed at q{ep['actual_quantile']:.2f}")
    print(f"sector_move : {sector_move(t, a)}")
    print(f"catalyst    : {known_catalyst(t, a)}")
    print(f"data_sanity : {data_sanity(t, a)}")
    print(f"--> {ep['classification']}  —  {ep['note']}\n")

for label in eps["classification"].unique():
    show(eps[eps["classification"] == label].iloc[0])

=== TSLA  2025-05-28  (momentum-breakout) ===
forecast median +2.35% | actual -1.65% | landed at q0.00
sector_move : {'name_ret': -0.0165, 'cluster_ret': -0.0035, 'peers_n': 28, 'moved_together': False, 'read': 'idiosyncratic'}
catalyst    : {'earnings': [], 'macro': [], 'has_earnings': False, 'has_macro': False}
data_sanity : {'ok': True, 'issues': [], 'gap': 0.0054}
--> unexplained  —  Idiosyncratic TSLA drop, no cluster move, no earnings/macro catalyst identified; cause unclear.

=== HD  2025-05-28  (quiet) ===
forecast median +1.03% | actual -0.63% | landed at q0.10
sector_move : {'name_ret': -0.0063, 'cluster_ret': -0.0052, 'peers_n': 25, 'moved_together': True, 'read': 'regime'}
catalyst    : {'earnings': [], 'macro': [], 'has_earnings': False, 'has_macro': False}
data_sanity : {'ok': True, 'issues': [], 'gap': -0.0043}
--> regime-move  —  cluster moved -0.52%, name -0.63% (no scheduled catalyst)

=== TXN  2025-05-28  (trend-continuation) ===
forecast median +1.33% | actual +0.50

## 2 · Catalyst recall — the planted-label test (PLAN §7)

On sessions that coincide with a scheduled earnings date, does the post-mortem
name the catalyst?

In [4]:
from eval.evaluate import postmortem_catalyst_recall
pd.Series(postmortem_catalyst_recall())

n                           6
named_catalyst              6
recall                    1.0
wilson95          [0.61, 1.0]
dtype: object

## 3 · Dispersion self-consistency

When it says *within-expected-dispersion*, did the close actually land
mid-distribution? When it names an event, was the actual in the tail?

In [5]:
from eval.evaluate import postmortem_dispersion_consistency
pd.Series(postmortem_dispersion_consistency())

n                             48.000
within_label_n                 9.000
within_label_median_|q-.5|     0.205
within_label_off_tail          1.000
event_label_n                 30.000
event_label_in_tail            0.933
dtype: float64

## 4 · The memory this writes

In [6]:
from src.memory import rebuild_stats, recall
stats = rebuild_stats()
print(len(stats), "(ticker, setup) keys in the derived stat table\n")
for ep in eps.head(6).itertuples():
    print(recall(ep.ticker, ep.setup).line())

41 (ticker, setup) keys in the derived stat table

desk has 1 prior momentum-breakout episode(s) for TSLA: direction hit 0%, mean signed error -4.00%
desk has 2 prior quiet episode(s) for HD: direction hit 50%, mean signed error -0.54%
desk has 1 prior trend-continuation episode(s) for DIS: direction hit 0%, mean signed error -1.68%
desk has 1 prior trend-continuation episode(s) for MS: direction hit 0%, mean signed error -2.29%
desk has 2 prior trend-continuation episode(s) for TXN: direction hit 50%, mean signed error +1.07%
desk has 2 prior trend-continuation episode(s) for META: direction hit 100%, mean signed error +0.73%


---
The `(ticker, setup)` stat lines above are exactly what the **next** morning's
triage `recall_memory` tool returns — the feedback edge that closes the loop.